# F01 — DOMINION : Le Prophète
> *"La voix est l'arme première. Purifiez-la avant de l'envoyer au combat."*
> — Ordre de la Rose Sacrée, Adepta Sororitas

```
╔══════════════════════════════════════════════════════════════╗
║   FRÉGATE F01 — DOMINION                                    ║
║   Rôle    : Clonage & Synthèse Vocale (OmniVoice)           ║
║   IN      : reference_vocale.wav + script_text.txt          ║
║   OUT     : voix_brute.wav                                  ║
║   Stack   : OmniVoice-Studio → httpx → Gradio headless      ║
╚══════════════════════════════════════════════════════════════╝
```

---

## Ordre des Cellules

| # | Cellule | Run | Description |
|---|---------|-----|-------------|
| 0 | **SETUP DRIVE** | **UNE SEULE FOIS** | Crée toute la structure Drive + liber_sanctorum.json |
| 1 | INIT | Chaque session | Monter Drive, cloner SANCTORUM, définir chemins |
| 2 | INSTALLATION | Chaque session | Installer OmniVoice-Studio + dépendances |
| 3 | SERVEUR | Chaque session | Lancer le backend OmniVoice en arrière-plan |
| 4 | INTERFACE | Chaque session | Interface Gradio — Le Prophète |
| 5 | SR_CUSTOS | Après génération | Check-in final vers le CMS flotte |

---
## CELLULE 0 — SETUP DRIVE
### ⚠️ RUN ONCE — EXÉCUTER UNE SEULE FOIS POUR TOUT LE PROJET
*Crée la structure Google Drive complète pour toute la flotte SANCTORUM.*
*N'exécuter qu'une seule fois. Idempotente : ne recrée pas ce qui existe déjà.*

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 0 — SETUP DRIVE SANCTORUM  (run once)          ║
# ║  Crée toute la structure Drive pour les 3 frégates       ║
# ╚══════════════════════════════════════════════════════════╝

import os
import json
from google.colab import drive

print("[SETUP] Montage de Google Drive...")
drive.mount('/content/drive', force_remount=False)

# ── Racine du projet sur Drive ──────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/SANCTORUM"

# ── Arborescence complète à créer ───────────────────────────────────────
STRUCTURE = [
    # Racine + tracking global
    f"{DRIVE_ROOT}/TRACKING",

    # F01 DOMINION — Le Prophète (Voice Cloning)
    f"{DRIVE_ROOT}/F01_DOMINION/IN",
    f"{DRIVE_ROOT}/F01_DOMINION/OUT",
    f"{DRIVE_ROOT}/F01_DOMINION/TRACKING",
    f"{DRIVE_ROOT}/F01_DOMINION/CODEBASE",

    # F02 CELESTIAN — La Montre (Purification DSP)
    f"{DRIVE_ROOT}/F02_CELESTIAN/IN",
    f"{DRIVE_ROOT}/F02_CELESTIAN/OUT",
    f"{DRIVE_ROOT}/F02_CELESTIAN/TRACKING",
    f"{DRIVE_ROOT}/F02_CELESTIAN/CODEBASE/presets",

    # F03 SERAPHIM — L'Architecte + La Machine à Micro-jets
    f"{DRIVE_ROOT}/F03_SERAPHIM/IN",
    f"{DRIVE_ROOT}/F03_SERAPHIM/OUT",
    f"{DRIVE_ROOT}/F03_SERAPHIM/TRACKING",
    f"{DRIVE_ROOT}/F03_SERAPHIM/CODEBASE",

    # SHARED — Bus inter-frégates
    f"{DRIVE_ROOT}/SHARED/IN",
    f"{DRIVE_ROOT}/SHARED/OUT",
]

# ── Création des dossiers ───────────────────────────────────────────────
created = []
skipped = []
for path in STRUCTURE:
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        created.append(path.replace(DRIVE_ROOT, "SANCTORUM"))
    else:
        skipped.append(path.replace(DRIVE_ROOT, "SANCTORUM"))

# ── Déposer liber_sanctorum.json sur Drive (CMS centralisé) ────────────
LIBER_DRIVE = f"{DRIVE_ROOT}/liber_sanctorum.json"
if not os.path.exists(LIBER_DRIVE):
    liber_data = {
        "_comment": "CMS CENTRALISÉ — LIBER SANCTORUM. Seul SR_CUSTOS.py est autorisé à modifier fleet_status.",
        "fleet_status": "pending_sanctification",
        "bpm": None,
        "f01_dominion": {
            "status": "idle",
            "script_text": "",
            "voice_reference": "",
            "output_path": ""
        },
        "f02_celestian": {
            "status": "idle",
            "preset_name": "standard_voix_purifiee",
            "apply_divine_presence": True,
            "compression_threshold_db": -15.0,
            "reverb_wet_level": 0.08,
            "output_path": ""
        },
        "f03_seraphim": {
            "status": "idle",
            "music_canvas": "",
            "directives_path": "",
            "output_path": ""
        },
        "sr_custos": {
            "last_validation": None,
            "errors": []
        },
        "final_output": ""
    }
    with open(LIBER_DRIVE, "w", encoding="utf-8") as f:
        json.dump(liber_data, f, indent=2, ensure_ascii=False)
    created.append("SANCTORUM/liber_sanctorum.json")
else:
    skipped.append("SANCTORUM/liber_sanctorum.json")

# ── Déposer les fichiers .gitkeep dans IN/OUT vides ─────────────────────
GITKEEP_DIRS = [
    f"{DRIVE_ROOT}/F01_DOMINION/IN/.gitkeep",
    f"{DRIVE_ROOT}/F01_DOMINION/OUT/.gitkeep",
    f"{DRIVE_ROOT}/F02_CELESTIAN/IN/.gitkeep",
    f"{DRIVE_ROOT}/F02_CELESTIAN/OUT/.gitkeep",
    f"{DRIVE_ROOT}/F03_SERAPHIM/IN/.gitkeep",
    f"{DRIVE_ROOT}/F03_SERAPHIM/OUT/.gitkeep",
    f"{DRIVE_ROOT}/SHARED/IN/.gitkeep",
    f"{DRIVE_ROOT}/SHARED/OUT/.gitkeep",
]
for gk in GITKEEP_DIRS:
    if not os.path.exists(gk):
        with open(gk, 'w') as f: f.write('')

# ── Rapport ────────────────────────────────────────────────────────────
print("\n╔══════════════════════════════════════════════════════╗")
print("║   SETUP DRIVE SANCTORUM — RAPPORT                    ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Racine : {DRIVE_ROOT:<43}║")
print(f"║  Créés  : {str(len(created)):<43}║")
print(f"║  Déjà existants : {str(len(skipped)):<34}║")
print("╠══════════════════════════════════════════════════════╣")
for path in created:
    print(f"║  + {path:<50}║")
print("╚══════════════════════════════════════════════════════╝")
print("\n[SETUP] Structure Drive SANCTORUM prête. Ne pas relancer cette cellule.")

---
## CELLULE 1 — INIT
*Monter Drive, cloner le dépôt SANCTORUM, définir tous les chemins.*

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — INIT                                       ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json
from google.colab import drive

# Monter Drive si pas encore monté
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# ── Chemins Drive ───────────────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/SANCTORUM"
F01_IN          = f"{DRIVE_ROOT}/F01_DOMINION/IN"
F01_OUT         = f"{DRIVE_ROOT}/F01_DOMINION/OUT"
F01_TRACKING    = f"{DRIVE_ROOT}/F01_DOMINION/TRACKING"
LIBER_DRIVE     = f"{DRIVE_ROOT}/liber_sanctorum.json"
GLOBAL_TRACKING = f"{DRIVE_ROOT}/TRACKING"

# ── Chemins locaux (Colab /content) ─────────────────────────────────────
COLAB_ROOT   = "/content"
SANCTORUM_DIR = f"{COLAB_ROOT}/SANCTORUM"
OMNIVOICE_DIR = f"{COLAB_ROOT}/OmniVoice-Studio"

# ── Cloner SANCTORUM si absent ───────────────────────────────────────────
if not os.path.exists(SANCTORUM_DIR):
    print("[INIT] Clonage du dépôt SANCTORUM...")
    !git clone https://github.com/kioka8877-ux/SANCTORUM.git {SANCTORUM_DIR} -q
else:
    print("[INIT] SANCTORUM déjà présent — pull...")
    !git -C {SANCTORUM_DIR} pull -q

# Ajouter au sys.path pour importer SR_CUSTOS
if SANCTORUM_DIR not in sys.path:
    sys.path.insert(0, SANCTORUM_DIR)

# ── Vérifier la structure Drive ──────────────────────────────────────────
for d in [F01_IN, F01_OUT, F01_TRACKING]:
    os.makedirs(d, exist_ok=True)

# ── Lire le liber ────────────────────────────────────────────────────────
if os.path.exists(LIBER_DRIVE):
    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    fleet_status = liber.get('fleet_status', 'unknown')
else:
    fleet_status = 'liber_not_found — exécuter cellule 0 en premier'

print(f"\n[INIT] Fleet status : {fleet_status}")
print(f"[INIT] Drive IN     : {F01_IN}")
print(f"[INIT] Drive OUT    : {F01_OUT}")
print("[INIT] Prêt.")

---
## CELLULE 2 — INSTALLATION OmniVoice
*Cloner et installer OmniVoice-Studio + dépendances F01.*

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — SALLE DES MACHINES                         ║
# ║  Installation épinglée + vérification + moteur F01       ║
# ║  C4 ne démarre QUE si F01_ENGINE.status == 'ready'       ║
# ╚══════════════════════════════════════════════════════════╝

import sys, os

_RAPPORT = []
F01_ENGINE = None

def _ok(msg):  _RAPPORT.append(f"[OK] {msg}"); print(f"  [OK] {msg}")
def _err(msg): _RAPPORT.append(f"[!!] {msg}"); print(f"  [!!] {msg}")


# ── ÉTAPE 1 : Torch triplet épinglé (avant tout import torch) ────────
print("\n── ÉTAPE 1 : Torch triplet épinglé ────────────────────────")
!pip install -q --upgrade pip
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

try:
    import torch, torchvision
    _ok(f"torch {torch.__version__} | torchvision {torchvision.__version__}")
except Exception as e:
    _err(f"torch import FAIL : {e}")

try:
    import torchvision.ops
    _ = torchvision.ops.nms
    _ok("torchvision::nms accessible")
except Exception as e:
    _err(f"torchvision::nms FAIL : {e}")


# ── ÉTAPE 2 : Dépendances backend ────────────────────────────────────
print("\n── ÉTAPE 2 : Dépendances backend ───────────────────────────")
!pip install -q fastapi "uvicorn[standard]" httpx python-multipart aiofiles pydantic soundfile librosa numpy scipy "gradio>=4.0"

try:
    import soundfile, librosa, gradio
    _ok("soundfile / librosa / gradio OK")
except Exception as e:
    _err(f"deps backend FAIL : {e}")


# ── ÉTAPE 3 : f5-tts avec contrainte de version ───────────────────────
print("\n── ÉTAPE 3 : f5-tts épinglé ────────────────────────────────")
!pip install -q f5-tts "torch==2.4.1+cu121" "torchvision==0.19.1+cu121" "torchaudio==2.4.1+cu121" --extra-index-url https://download.pytorch.org/whl/cu121

_F5TTS_cls = None

try:
    from transformers import pipeline as _tp
    _ok("transformers.pipeline OK")
except Exception as e:
    _err(f"transformers.pipeline FAIL : {e}")

try:
    from f5_tts.api import F5TTS as _F5TTS_cls
    _ok("f5_tts.api OK")
except Exception as e:
    _err(f"f5_tts.api FAIL : {e}")
    _F5TTS_cls = None


# ── ÉTAPE 4 : Pré-chargement modèle F5TTS ────────────────────────────
print("\n── ÉTAPE 4 : Pré-chargement modèle F5TTS ───────────────────")
_f5_model_c2 = None
if _F5TTS_cls is not None:
    try:
        print("[C2] Chargement F5TTS (2-3 min au 1er lancement)...")
        _f5_model_c2 = _F5TTS_cls()
        _ok("Modèle F5TTS chargé")
    except Exception as e:
        _err(f"F5TTS load FAIL : {e}")
else:
    _err("F5TTS class absente — modèle non chargé")


# ── ÉTAPE 5 : Probe OmniVoice ─────────────────────────────────────────
print("\n── ÉTAPE 5 : Probe OmniVoice ───────────────────────────────")
import httpx as _httpx
_OMNIVOICE_PORT = 8080
_omnivoice_alive = False
for _p in ["/health", "/", "/docs"]:
    try:
        _r = _httpx.get(f"http://localhost:{_OMNIVOICE_PORT}{_p}", timeout=2)
        if _r.status_code < 500:
            _omnivoice_alive = True
            break
    except Exception:
        pass
if _omnivoice_alive:
    _ok(f"OmniVoice actif port {_OMNIVOICE_PORT}")
else:
    _ok("OmniVoice absent — mode f5-tts direct actif")


# ── CONSTRUCTION F01_ENGINE ───────────────────────────────────────────
print("\n── Construction F01_ENGINE ─────────────────────────────────")
import soundfile as _sf_engine


def _engine_generate(script, ref_path, ref_text, speed):
    """Génération vocale — appelée par C4 via F01_ENGINE['generate']."""
    if not ref_path or not os.path.exists(ref_path):
        raise ValueError(
            "Référence vocale manquante.\n"
            "F5-TTS requiert un fichier de référence.\n"
            "→ Uploadez un fichier dans l'onglet 'Référence Drive'."
        )
    result = _f5_model_c2.infer(
        ref_file=ref_path,
        ref_text=ref_text or "",
        gen_text=script,
        speed=speed,
    )
    if isinstance(result, (tuple, list)) and len(result) >= 2:
        return result[0], result[1]
    raise RuntimeError(f"F5TTS.infer() retour inattendu : {type(result).__name__}")


def _engine_save(wav, sr, out_path):
    """Sauvegarde audio — appelée par C4 via F01_ENGINE['save']."""
    _sf_engine.write(out_path, wav, sr)


if _f5_model_c2 is not None:
    F01_ENGINE = {
        "mode"      : "omnivoice-api" if _omnivoice_alive else "f5-tts",
        "model"     : _f5_model_c2,
        "server_url": f"http://localhost:{_OMNIVOICE_PORT}" if _omnivoice_alive else None,
        "generate"  : _engine_generate,
        "save"      : _engine_save,
        "status"    : "ready",
    }
    _ok("F01_ENGINE prêt — statut VERT")
else:
    F01_ENGINE = None
    _err("F01_ENGINE = None — C4 ne peut pas démarrer")


# ── RAPPORT FINAL ─────────────────────────────────────────────────────
print("\n╔══════════════════════════════════════════════════════════╗")
print("║    RAPPORT SALLE DES MACHINES — F01 DOMINION             ║")
print("╠══════════════════════════════════════════════════════════╣")
for _line in _RAPPORT:
    _sym = "OK" if _line.startswith("[OK]") else "!!"
    _body = _line[5:]
    print(f"║  [{_sym}] {_body:<50}║")
print("╠══════════════════════════════════════════════════════════╣")
_global_status = "VERT — C4 peut demarrer" if F01_ENGINE is not None else "ROUGE — relancer C2"
print(f"║  STATUT : {_global_status:<47}║")
print("╚══════════════════════════════════════════════════════════╝")


---
## CELLULE 3 — SERVEUR OmniVoice
*Lancer le backend OmniVoice en arrière-plan sur le port 8080.*

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — SERVEUR OmniVoice Backend                  ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess, time, os, httpx

OMNIVOICE_DIR  = "/content/OmniVoice-Studio"
OMNIVOICE_PORT = 8080


def omnivoice_alive():
    """Sonde /health, / et /docs — robuste aux backends sans endpoint /health."""
    for path in ["/health", "/", "/docs"]:
        try:
            r = httpx.get(f"http://localhost:{OMNIVOICE_PORT}{path}", timeout=2)
            if r.status_code < 500:
                return True
        except Exception:
            pass
    return False


def _find_backend(base_dir: str):
    """
    Détecte dynamiquement le point d'entrée uvicorn.
    Cherche main:app / app:app dans backend/ puis à la racine.
    """
    candidates = [
        (f"{base_dir}/backend", "main:app"),
        (f"{base_dir}/backend", "app:app"),
        (base_dir,              "main:app"),
        (base_dir,              "app:app"),
        (base_dir,              "server:app"),
    ]
    for cwd, module in candidates:
        py_file = os.path.join(cwd, module.split(":")[0] + ".py")
        if os.path.exists(py_file):
            print(f"[SERVER] Point d'entrée détecté : {py_file}")
            return cwd, module
    return None, None


if omnivoice_alive():
    print(f"[SERVER] OmniVoice déjà actif sur port {OMNIVOICE_PORT}.")
else:
    BACKEND_DIR, APP_MODULE = _find_backend(OMNIVOICE_DIR)

    if BACKEND_DIR is None:
        print("[SERVER] Structure OmniVoice non reconnue — passage en mode f5-tts direct.")
        if os.path.exists(OMNIVOICE_DIR):
            print(f"[SERVER]   Contenu {OMNIVOICE_DIR}/: {sorted(os.listdir(OMNIVOICE_DIR))[:10]}")
        else:
            print(f"[SERVER]   {OMNIVOICE_DIR} absent — vérifier cellule 2.")
    else:
        os.makedirs("/content/omnivoice_data/voices",  exist_ok=True)
        os.makedirs("/content/omnivoice_data/outputs", exist_ok=True)

        env = os.environ.copy()
        env.update({
            "VOICES_DIR"  : "/content/omnivoice_data/voices",
            "OUTPUTS_DIR" : "/content/omnivoice_data/outputs",
            "PORT"        : str(OMNIVOICE_PORT),
            "HOST"        : "0.0.0.0",
        })

        print(f"[SERVER] Lancement OmniVoice ({APP_MODULE}) sur port {OMNIVOICE_PORT}...")
        proc = subprocess.Popen(
            ["python", "-m", "uvicorn", APP_MODULE,
             "--host", "0.0.0.0",
             "--port", str(OMNIVOICE_PORT),
             "--log-level", "warning"],
            cwd=BACKEND_DIR,
            env=env,
            stdout=open("/content/omnivoice.log", "w"),
            stderr=subprocess.STDOUT,
        )

        # F5-TTS peut prendre 3-5 min au 1er téléchargement — max 5 min
        print("[SERVER] Attente démarrage (max 5 min)", end="")
        for _ in range(60):
            time.sleep(5)
            print(".", end="", flush=True)
            if omnivoice_alive():
                break
        print()

        if omnivoice_alive():
            print(f"[SERVER] OmniVoice actif — http://localhost:{OMNIVOICE_PORT}")
            try:
                r = httpx.get(f"http://localhost:{OMNIVOICE_PORT}/engines", timeout=5)
                print(f"[SERVER] Moteurs : {[e.get('id') for e in r.json()]}")
            except Exception:
                pass
        else:
            print("[SERVER] Délai dépassé (5 min) — mode f5-tts direct.")
            print("[SERVER] Consulter les erreurs : /content/omnivoice.log")

OMNIVOICE_SERVER_URL = f"http://localhost:{OMNIVOICE_PORT}"
USE_SERVER = omnivoice_alive()
print(f"[SERVER] Mode actif : {'API HTTP (OmniVoice)' if USE_SERVER else 'f5-tts direct'}")


---
## CELLULE 4 — INTERFACE GRADIO — LE PROPHÈTE
*Interface opérateur headless. Le navigateur est le seul terminal.*

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — SALLE DE BAL — LE PROPHÈTE                 ║
# ║  Interface Gradio legere — le courant vient de C2        ║
# ║  Seules pannes possibles : fichier manquant, script vide ║
# ╚══════════════════════════════════════════════════════════╝

# ── Vérification : salle des machines prête ? ─────────────────────────
_c4_ok = (
    isinstance(globals().get('F01_ENGINE'), dict)
    and globals().get('F01_ENGINE', {}).get('status') == 'ready'
)
if not _c4_ok:
    print("╔══════════════════════════════════════════════════════════╗")
    print("║  SALLE DES MACHINES HORS LIGNE                           ║")
    print("║  F01_ENGINE non disponible — relancer C2 d'abord         ║")
    print("╚══════════════════════════════════════════════════════════╝")

if _c4_ok:
    import os, json, httpx, shutil, hashlib, time, tempfile
    from datetime import datetime, timezone
    import gradio as gr

    # ── Chemins Drive ────────────────────────────────────────────────────
    DRIVE_ROOT   = "/content/drive/MyDrive/SANCTORUM"
    F01_IN       = f"{DRIVE_ROOT}/F01_DOMINION/IN"
    F01_OUT      = f"{DRIVE_ROOT}/F01_DOMINION/OUT"
    F01_TRACKING = f"{DRIVE_ROOT}/F01_DOMINION/TRACKING"
    LIBER_DRIVE  = f"{DRIVE_ROOT}/liber_sanctorum.json"

    _MODE = F01_ENGINE["mode"]
    print(f"[C4] Moteur connecte — mode : {_MODE}")

    # ── Fonctions utilitaires ─────────────────────────────────────────────

    def list_drive_references():
        exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
        if os.path.exists(F01_IN):
            files = [f for f in os.listdir(F01_IN) if os.path.splitext(f)[1].lower() in exts]
            return files or ["(aucun fichier — uploader via l'onglet Reference)"]
        return ["(aucun fichier — uploader via l'onglet Reference)"]

    def upload_reference_to_drive(file_obj):
        if file_obj is None:
            return "Aucun fichier selectionne.", gr.update()
        file_path = file_obj if isinstance(file_obj, str) else file_obj.name
        dest = os.path.join(F01_IN, os.path.basename(file_path))
        shutil.copy(file_path, dest)
        refs = list_drive_references()
        return f"Reference deposee : {dest}", gr.update(choices=refs, value=refs[0])

    def _log_f01(event, detail=""):
        log_path = os.path.join(F01_TRACKING, "F01_LOG.md")
        ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"\n## [{ts}] {event}\n{detail}\n")

    def _update_liber(voice_ref_name, output_path):
        if not os.path.exists(LIBER_DRIVE):
            return
        with open(LIBER_DRIVE, "r", encoding="utf-8") as f:
            liber = json.load(f)
        liber["f01_dominion"]["status"]          = "done"
        liber["f01_dominion"]["voice_reference"] = voice_ref_name
        liber["f01_dominion"]["output_path"]     = output_path
        liber["fleet_status"]                    = "voice_raw_ready"
        liber["sr_custos"]["last_validation"]    = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        with open(LIBER_DRIVE, "w", encoding="utf-8") as f:
            json.dump(liber, f, indent=2, ensure_ascii=False)

    def _generate_via_server(script, ref_path, ref_text, language, speed, effect):
        """Appeler l'API OmniVoice — uniquement si F01_ENGINE mode omnivoice-api."""
        server_url = F01_ENGINE["server_url"]
        lang = language if language not in ("Auto", "", None) else None
        profile_id = None
        if ref_path and os.path.exists(ref_path):
            profile_data = {"name": f"dominion_{int(time.time())}", "ref_text": ref_text}
            if lang:
                profile_data["language"] = lang
            with open(ref_path, 'rb') as rf:
                resp = httpx.post(
                    f"{server_url}/profiles",
                    data=profile_data,
                    files={"ref_audio": (os.path.basename(ref_path), rf, "audio/wav")},
                    timeout=60
                )
                if resp.status_code == 200:
                    profile_id = resp.json().get("id")
        payload = {
            "model": "omnivoice", "input": script,
            "voice": profile_id or "default",
            "response_format": "wav", "speed": speed,
        }
        if lang:
            payload["language"] = lang
        if effect and effect != "none":
            payload["effect"] = effect
        with httpx.Client(timeout=300) as client:
            resp = client.post(f"{server_url}/v1/audio/speech", json=payload)
        if resp.status_code != 200:
            raise RuntimeError(f"OmniVoice API error {resp.status_code}: {resp.text[:200]}")
        return resp.content

    # ── Fonction principale de génération ─────────────────────────────────

    def generate_voice(script, ref_choice, ref_upload, ref_text, language, speed, effect):
        if not script.strip():
            return None, "[ERREUR] Le script est vide."

        ref_path, ref_name = None, "none"
        if ref_upload is not None:
            ref_path = ref_upload if isinstance(ref_upload, str) else ref_upload.name
            ref_name = os.path.basename(ref_path)
        elif ref_choice and "(aucun fichier" not in ref_choice:
            ref_path = os.path.join(F01_IN, ref_choice)
            ref_name = ref_choice

        status_log = [
            f"[F01] Reference : {ref_name}",
            f"[F01] Mode      : {_MODE}",
            f"[F01] Langue    : {language} | Vitesse : {speed}",
            "[F01] Synthese en cours...",
        ]
        tmp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        try:
            t0 = time.time()
            if _MODE == "omnivoice-api":
                try:
                    audio_bytes = _generate_via_server(script, ref_path, ref_text, language, speed, effect)
                    tmp_wav.write(audio_bytes)
                    tmp_wav.close()
                    out_tmp = tmp_wav.name
                except RuntimeError as srv_err:
                    if "404" in str(srv_err):
                        status_log.append("[F01] OmniVoice 404 → bascule f5-tts")
                        tmp_wav.close()
                        wav, sr = F01_ENGINE["generate"](script, ref_path, ref_text or "", speed)
                        F01_ENGINE["save"](wav, sr, tmp_wav.name)
                        out_tmp = tmp_wav.name
                    else:
                        raise
            else:
                tmp_wav.close()
                wav, sr = F01_ENGINE["generate"](script, ref_path, ref_text or "", speed)
                F01_ENGINE["save"](wav, sr, tmp_wav.name)
                out_tmp = tmp_wav.name

            elapsed = round(time.time() - t0, 1)
            status_log.append(f"[F01] Generation terminee en {elapsed}s")

            out_drive = os.path.join(F01_OUT, "voix_brute.wav")
            shutil.copy(out_tmp, out_drive)
            status_log.append(f"[F01] Depose sur Drive : {out_drive}")

            h = hashlib.md5()
            with open(out_drive, "rb") as fp:
                for chunk in iter(lambda: fp.read(8192), b""):
                    h.update(chunk)
            status_log.append(f"[F01] MD5 : {h.hexdigest()}")

            _update_liber(ref_name, out_drive)
            status_log.append("[F01] liber mis a jour — fleet_status: voice_raw_ready")
            _log_f01("SYNTHESE TERMINEE",
                     f"ref={ref_name} | lang={language} | speed={speed} | elapsed={elapsed}s")
            return out_tmp, "\n".join(status_log)

        except Exception as e:
            import traceback
            status_log.extend([f"[ERREUR] {e}", traceback.format_exc()])
            _log_f01("ERREUR", str(e))
            if not tmp_wav.closed:
                tmp_wav.close()
            return None, "\n".join(status_log)

    # ╔══════════════════════════════════════════════════════════╗
    # ║  INTERFACE GRADIO                                        ║
    # ╚══════════════════════════════════════════════════════════╝

    CSS = """
    .gradio-container { background: #0d0d0d; }
    #sanctorum-header { text-align: center; padding: 16px; border: 1px solid #4a1a1a;
        background: linear-gradient(135deg, #1a0505 0%, #0d0d0d 100%); margin-bottom: 12px; }
    #sanctorum-header h1 { color: #c0392b; font-family: monospace; font-size: 1.4em; }
    #sanctorum-header p  { color: #888; font-size: 0.85em; font-family: monospace; }
    .label-text { color: #aaa !important; font-family: monospace !important; }
    button.primary { background: #7b1c1c !important; color: #f8d7da !important; }
    """

    with gr.Blocks(css=CSS, title="F01 DOMINION — Le Prophete") as demo:

        gr.HTML("""
        <div id='sanctorum-header'>
          <h1>F01 — DOMINION : Le Prophete</h1>
          <p>Fregate SANCTORUM &middot; Clonage &amp; Synthese Vocale &middot; OmniVoice-Studio</p>
          <p>Ordre de la Rose Sacree — Adepta Sororitas</p>
        </div>
        """)

        with gr.Tabs():

            with gr.Tab("Synthese Vocale"):
                with gr.Row():
                    with gr.Column(scale=3):
                        script_input = gr.Textbox(
                            label="Script a synthetiser",
                            placeholder="Entrez ici le texte a faire prononcer par la voix clonee...",
                            lines=8, elem_classes="label-text"
                        )
                        ref_text_input = gr.Textbox(
                            label="Transcription de la voix de reference (optionnel)",
                            placeholder="Texte dit dans le fichier audio de reference...",
                            lines=2, elem_classes="label-text"
                        )
                    with gr.Column(scale=2):
                        ref_dropdown = gr.Dropdown(
                            label="Reference vocale (Drive F01_DOMINION/IN/)",
                            choices=list_drive_references(),
                            elem_classes="label-text"
                        )
                        ref_upload = gr.File(
                            label="Ou uploader une nouvelle reference (.wav / .mp3)",
                            file_types=[".wav", ".mp3", ".flac", ".ogg", ".m4a"]
                        )
                        upload_btn = gr.Button("Deposer sur Drive", size="sm")
                        upload_status = gr.Textbox(label="", lines=1, interactive=False, show_label=False)

                with gr.Row():
                    language_input = gr.Dropdown(
                        label="Langue",
                        choices=["Auto", "fr", "en", "es", "de", "it", "pt", "zh", "ja", "ko", "ar"],
                        value="Auto", scale=1
                    )
                    speed_input = gr.Slider(
                        label="Vitesse", minimum=0.5, maximum=2.0, step=0.05, value=1.0, scale=2
                    )
                    effect_input = gr.Dropdown(
                        label="Preset d'effet (OmniVoice)",
                        choices=["broadcast", "clean", "studio", "telephone", "none"],
                        value="broadcast", scale=1
                    )

                generate_btn = gr.Button("SYNTHETISER — DOMINION", variant="primary", size="lg")

                with gr.Row():
                    audio_output = gr.Audio(
                        label="voix_brute.wav (sortie F01 → F01_DOMINION/OUT/)",
                        type="filepath", scale=3
                    )
                    status_output = gr.Textbox(
                        label="Journal de mission", lines=12, interactive=False, scale=2
                    )

            with gr.Tab("Reference Drive"):
                gr.Markdown("""
                ### Gestion des references vocales — `SANCTORUM/F01_DOMINION/IN/`
                Uploadez vos fichiers de reference ici. Ils seront copies sur Google Drive
                et disponibles dans le selecteur de l'onglet **Synthese Vocale**.
                """)
                ref_upload_2 = gr.File(
                    label="Uploader un fichier de reference vocale",
                    file_types=[".wav", ".mp3", ".flac", ".ogg", ".m4a"]
                )
                upload_btn_2 = gr.Button("Deposer sur Drive", variant="primary")
                upload_status_2 = gr.Textbox(label="Statut", lines=2, interactive=False)
                ref_list = gr.Textbox(
                    label="References disponibles dans F01_DOMINION/IN/",
                    value="\n".join(list_drive_references()),
                    lines=8, interactive=False
                )
                refresh_btn = gr.Button("Rafraichir la liste")

            with gr.Tab("Statut Flotte"):
                gr.Markdown("### liber_sanctorum.json — Statut en temps reel")
                liber_display = gr.JSON(label="liber_sanctorum.json")
                read_liber_btn = gr.Button("Lire le Liber")

        # ── Cablage des evenements ─────────────────────────────────────────

        upload_btn.click(
            upload_reference_to_drive, inputs=[ref_upload], outputs=[upload_status, ref_dropdown]
        )
        upload_btn_2.click(
            upload_reference_to_drive, inputs=[ref_upload_2], outputs=[upload_status_2, ref_dropdown]
        ).then(
            lambda: "\n".join(list_drive_references()), inputs=[], outputs=[ref_list]
        )
        refresh_btn.click(
            lambda: ("\n".join(list_drive_references()), gr.update(choices=list_drive_references())),
            inputs=[], outputs=[ref_list, ref_dropdown]
        )
        read_liber_btn.click(
            lambda: json.load(open(LIBER_DRIVE)) if os.path.exists(LIBER_DRIVE) else {"error": "liber non trouve"},
            inputs=[], outputs=[liber_display]
        )
        generate_btn.click(
            generate_voice,
            inputs=[script_input, ref_dropdown, ref_upload, ref_text_input,
                    language_input, speed_input, effect_input],
            outputs=[audio_output, status_output]
        )

    print("[GRADIO] Demarrage interface Le Prophete...")
    demo.launch(share=True, debug=True, server_port=7860, inbrowser=False)


---
## CELLULE 5 — SR_CUSTOS CHECK-IN
*Valider la sortie F01 dans le CMS de flotte. Exécuter après la génération.*

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — SR_CUSTOS CHECK-IN                         ║
# ║  Valider la sortie F01 dans liber_sanctorum.json         ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil

DRIVE_ROOT    = "/content/drive/MyDrive/SANCTORUM"
SANCTORUM_DIR = "/content/SANCTORUM"
LIBER_DRIVE   = f"{DRIVE_ROOT}/liber_sanctorum.json"
OUT_PATH      = f"{DRIVE_ROOT}/F01_DOMINION/OUT/voix_brute.wav"

# ── Vérifier que la sortie existe ───────────────────────────────────────
if not os.path.exists(OUT_PATH):
    print(f"[SR_CUSTOS] ERREUR: voix_brute.wav non trouvé dans F01_DOMINION/OUT/")
    print(f"[SR_CUSTOS] Veuillez d'abord générer une synthèse dans la cellule 4.")
else:
    # ── Check-in via SR_CUSTOS.py (dans le repo cloné) ──────────────────
    if SANCTORUM_DIR not in sys.path:
        sys.path.insert(0, SANCTORUM_DIR)

    custos_script = os.path.join(SANCTORUM_DIR, "SR_CUSTOS.py")

    if os.path.exists(custos_script):
        local_liber = os.path.join(SANCTORUM_DIR, "liber_sanctorum.json")
        shutil.copy(LIBER_DRIVE, local_liber)

        !python {custos_script} --mode check-in --frigate F01 --output {OUT_PATH} 2>&1

        # Récupérer le liber mis à jour vers Drive
        shutil.copy(local_liber, LIBER_DRIVE)

    # ── Sync logs CUSTOS vers Drive (M-04) ──────────────────────────────
    for _log_name in ['SR_CAMPAIGN_LOG.md', 'SR_TRANSFER_LOG.md']:
        _log_src = os.path.join(SANCTORUM_DIR, 'TRACKING', _log_name)
        if os.path.exists(_log_src):
            shutil.copy(_log_src, os.path.join(DRIVE_ROOT, 'TRACKING', _log_name))
        else:
            print(f"[SR_CUSTOS] Log non trouvé : {_log_name} — sync ignoré.")

    # ── Afficher le statut final ─────────────────────────────────────────
    if os.path.exists(LIBER_DRIVE):
        with open(LIBER_DRIVE) as f:
            liber = json.load(f)
        print("\n╔══════════════════════════════════════════════════════╗")
        print("║          ÉTAT DE LA FLOTTE — POST F01                 ║")
        print("╠══════════════════════════════════════════════════════╣")
        print(f"║  fleet_status : {liber.get('fleet_status', 'n/a'):<36}║")
        print(f"║  F01 DOMINION : {liber['f01_dominion']['status']:<36}║")
        print(f"║  F02 CELESTIAN: {liber['f02_celestian']['status']:<36}║")
        print(f"║  F03 SERAPHIM : {liber['f03_seraphim']['status']:<36}║")
        print("╠══════════════════════════════════════════════════════╣")
        print("║  PROCHAINE ÉTAPE : F02_CELESTIAN (Purification DSP)  ║")
        print("╚══════════════════════════════════════════════════════╝")
